### CELL 1: Environment Configuration, Imports & Flask Init


In [11]:
import os
import sqlite3
import math
import secrets
from flask import Flask, render_template, request, redirect, url_for, session, flash, jsonify
from werkzeug.security import generate_password_hash, check_password_hash
import threading

app = Flask(__name__, template_folder='templates', static_folder='static')
app.secret_key = secrets.token_hex(32)

print("✅ Cell 1 Executed: Flask initialized.")


✅ Cell 1 Executed: Flask initialized.


### CELL 2: Cryptographic Security & CAPTCHA System


In [12]:
import random

def hash_password(password):
    return generate_password_hash(password, method='pbkdf2:sha256')

def verify_password(hashed, password):
    return check_password_hash(hashed, password)

def generate_captcha():
    num1 = random.randint(1, 10)
    num2 = random.randint(1, 10)
    operator = random.choice(['+', '*', '-'])
    if operator == '+':
        ans = num1 + num2
    elif operator == '*':
        ans = num1 * num2
    else:
        ans = num1 - num2
    
    question = f"{num1} {operator} {num2}"
    return question, str(ans)

print("✅ Cell 2 Executed: Security and CAPTCHA loaded.")


✅ Cell 2 Executed: Security and CAPTCHA loaded.


### CELL 3: Relational Database Schema & Migrations (SQL)


In [13]:
DATABASE = 'ecps_database.db'

def get_db():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db()
    c = conn.cursor()
    c.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        role TEXT CHECK(role IN ('STUDENT', 'ADMIN')) NOT NULL DEFAULT 'STUDENT',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    c.execute('''
    CREATE TABLE IF NOT EXISTS complaints (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER NOT NULL,
        title TEXT NOT NULL,
        category TEXT CHECK(category IN (
            'ELECTRICITY', 'WATER', 'CLASSROOM', 'TRANSPORT', 'INTERNET'
        )) NOT NULL,
        location TEXT NOT NULL,
        severity INTEGER CHECK(severity BETWEEN 1 AND 5) NOT NULL,
        affected_count INTEGER NOT NULL,
        safety_impact INTEGER CHECK(safety_impact BETWEEN 0 AND 4) NOT NULL,
        duration_hours INTEGER NOT NULL,
        is_recurring INTEGER CHECK(is_recurring IN (0, 1)) DEFAULT 0,
        description TEXT NOT NULL,
        
        priority_score REAL NOT NULL,
        priority_level TEXT CHECK(priority_level IN ('CRITICAL', 'HIGH', 'MEDIUM', 'LOW')) NOT NULL,
        assigned_department TEXT NOT NULL,
        explanation_summary TEXT NOT NULL,
        
        status TEXT CHECK(status IN ('LOGGED', 'IN_TRIAGE', 'DISPATCHED', 'RESOLVED')) DEFAULT 'LOGGED',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
    )
    ''')
    
    c.execute('CREATE INDEX IF NOT EXISTS idx_priority ON complaints(priority_score DESC)')
    c.execute('CREATE INDEX IF NOT EXISTS idx_status ON complaints(status)')
    
    # Seed Admin User
    c.execute("SELECT id FROM users WHERE role='ADMIN' LIMIT 1")
    if not c.fetchone():
        admin_pass = hash_password('admin123')
        c.execute("INSERT INTO users (username, email, password_hash, role) VALUES (?, ?, ?, ?)", 
                  ('admin', 'admin@university.edu', admin_pass, 'ADMIN'))
    
    conn.commit()
    conn.close()

init_db()
print("✅ Cell 3 Executed: Database initialized successfully.")


✅ Cell 3 Executed: Database initialized successfully.


### CELL 4: Rule Engine & Contradiction Detection Pipeline


In [14]:
CATEGORY_WEIGHTS = {
    'ELECTRICITY': {'weight': 1.40, 'dept': 'Electrical Maintenance Cell', 'max_safety': 4, 'sla': '2 Hours'},
    'WATER': {'weight': 1.35, 'dept': 'Public Health & Sanitation', 'max_safety': 3, 'sla': '3 Hours'},
    'CLASSROOM': {'weight': 1.15, 'dept': 'Estate & Civil Works', 'max_safety': 4, 'sla': '6 Hours'},
    'TRANSPORT': {'weight': 1.05, 'dept': 'Logistics & Transport Fleet', 'max_safety': 2, 'sla': '4 Hours'},
    'INTERNET': {'weight': 0.85, 'dept': 'IT Infrastructure Services', 'max_safety': 1, 'sla': '12 Hours'}
}

def validate_complaint(category, severity, headcount, safety_impact, location, description):
    # Rule 1: Safety vs Category Contradiction
    max_safe = CATEGORY_WEIGHTS[category]['max_safety']
    if safety_impact > max_safe:
        return False, f"Contradiction: {category} issues cannot have a safety impact of {safety_impact}. Maximum allowed is {max_safe}."
        
    # Rule 2: Headcount vs Location (Simplified check: Classroom single unit > 150)
    if 'Classroom' in location and headcount > 150:
        return False, "Contradiction: A standard classroom capacity prevents a localized defect from directly affecting over 150 individuals."
        
    # Rule 3: Severity vs Description
    if severity == 5 and len(description.strip()) < 10:
        return False, "Contradiction: Critical severity (5) requires a detailed description."
        
    # Rule 4: Zero-Impact High-Severity Paradox
    if severity >= 4 and headcount == 0 and safety_impact == 0:
        return False, "Contradiction: High severity cannot have zero affected headcount and zero safety impact."
        
    return True, "Valid"

print("✅ Cell 4 Executed: Rule Engine loaded.")


✅ Cell 4 Executed: Rule Engine loaded.


### CELL 5: Mathematical Prioritization & Explainability Engine


In [15]:
# Internal parameter weights
W_S = 1.0
W_N = 1.0
W_H = 2.0
W_D = 0.5

def compute_priority(category, severity, headcount, safety_impact, duration, is_recurring):
    base_weight = CATEGORY_WEIGHTS[category]['weight']
    phi_n = min(5.0, math.log10(headcount + 1))
    
    R = 1.25 if is_recurring else 1.0
    
    score = (base_weight * ((W_S * severity) + (W_N * phi_n) + (W_H * safety_impact) + (W_D * duration))) * R
    return score

def get_priority_level(score):
    if score >= 28.0: return 'CRITICAL'
    if score >= 20.0: return 'HIGH'
    if score >= 12.0: return 'MEDIUM'
    return 'LOW'

def generate_explanation(category, score, severity, headcount, safety, duration, is_recurring):
    dept = CATEGORY_WEIGHTS[category]['dept']
    sla = CATEGORY_WEIGHTS[category]['sla']
    
    exp = f"Calculated score: {score:.2f}. "
    if safety >= 3:
        exp += f"Primary driver: Severe {category} Hazard. "
    elif headcount > 50:
        exp += f"Primary driver: High impact scale ({headcount} affected). "
    else:
        exp += f"Primary driver: Severity {severity} issue in {category}. "
        
    if is_recurring:
        exp += "Score amplified by +25% due to recurring failure. "
        
    exp += f"Routed to {dept} with an SLA target of {sla}."
    return exp

def compare_issues(issue_a, issue_b):
    diff = issue_a['priority_score'] - issue_b['priority_score']
    
    if diff > 0:
        winner = issue_a
        loser = issue_b
    else:
        winner = issue_b
        loser = issue_a
        
    rationale = (
        f"Issue #{winner['id']} ({winner['title']}) is prioritized above Issue #{loser['id']} ({loser['title']}) "
        f"because it carries a Category weight of {CATEGORY_WEIGHTS[winner['category']]['weight']} "
        f"and Safety Impact factor H={winner['safety_impact']}, compared to H={loser['safety_impact']} "
        f"for Issue #{loser['id']}."
    )
    return rationale

print("✅ Cell 5 Executed: Math & Explainability Engine loaded.")


✅ Cell 5 Executed: Math & Explainability Engine loaded.


### CELL 6: Flask Route Handlers (Auth & Student Operations)


In [16]:
def login_required(f):
    def wrap(*args, **kwargs):
        if 'user_id' not in session:
            return redirect(url_for('login'))
        return f(*args, **kwargs)
    wrap.__name__ = f.__name__
    return wrap

@app.route('/captcha')
def captcha():
    q, a = generate_captcha()
    session['captcha_ans'] = a
    return jsonify({"question": q})

@app.route('/')
def index():
    if 'user_id' in session:
        return redirect(url_for('dashboard'))
    return redirect(url_for('login'))

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        username = request.form['username']
        password = request.form['password']
        captcha_ans = request.form.get('captcha')
        
        if captcha_ans != session.get('captcha_ans'):
            flash("Invalid CAPTCHA.")
            return redirect(url_for('login'))
            
        conn = get_db()
        c = conn.cursor()
        c.execute("SELECT * FROM users WHERE username = ?", (username,))
        user = c.fetchone()
        conn.close()
        
        if user and verify_password(user['password_hash'], password):
            session['user_id'] = user['id']
            session['role'] = user['role']
            session['username'] = user['username']
            if user['role'] == 'ADMIN':
                return redirect(url_for('admin_triage'))
            return redirect(url_for('dashboard'))
        else:
            flash("Invalid credentials.")
    return render_template('login.html')

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        username = request.form['username']
        email = request.form['email']
        password = request.form['password']
        
        conn = get_db()
        c = conn.cursor()
        try:
            c.execute("INSERT INTO users (username, email, password_hash, role) VALUES (?, ?, ?, 'STUDENT')",
                      (username, email, hash_password(password)))
            conn.commit()
            flash("Registration successful. Please login.")
            return redirect(url_for('login'))
        except sqlite3.IntegrityError:
            flash("Username or Email already exists.")
        finally:
            conn.close()
            
    return render_template('register.html')

@app.route('/logout')
def logout():
    session.clear()
    return redirect(url_for('login'))

@app.route('/dashboard', methods=['GET', 'POST'])
@login_required
def dashboard():
    if session['role'] == 'ADMIN':
        return redirect(url_for('admin_triage'))
        
    conn = get_db()
    c = conn.cursor()
    
    if request.method == 'POST':
        title = request.form['title']
        category = request.form['category']
        location = request.form['location']
        severity = int(request.form['severity'])
        headcount = int(request.form['affected_count'])
        safety = int(request.form['safety_impact'])
        duration = int(request.form['duration_hours'])
        recurring = 1 if request.form.get('is_recurring') else 0
        desc = request.form['description']
        
        is_valid, msg = validate_complaint(category, severity, headcount, safety, location, desc)
        if not is_valid:
            flash(msg)
        else:
            score = compute_priority(category, severity, headcount, safety, duration, recurring)
            level = get_priority_level(score)
            dept = CATEGORY_WEIGHTS[category]['dept']
            explanation = generate_explanation(category, score, severity, headcount, safety, duration, recurring)
            
            c.execute('''
                INSERT INTO complaints 
                (user_id, title, category, location, severity, affected_count, safety_impact, duration_hours, is_recurring, description, priority_score, priority_level, assigned_department, explanation_summary)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (session['user_id'], title, category, location, severity, headcount, safety, duration, recurring, desc, score, level, dept, explanation))
            conn.commit()
            flash("Complaint submitted successfully.")
            return redirect(url_for('dashboard'))
            
    c.execute("SELECT * FROM complaints WHERE user_id = ? ORDER BY created_at DESC", (session['user_id'],))
    complaints = c.fetchall()
    conn.close()
    
    return render_template('student_dashboard.html', complaints=complaints)

print("✅ Cell 6 Executed: Student Auth & Operations mapped.")


✅ Cell 6 Executed: Student Auth & Operations mapped.


### CELL 7: Flask Route Handlers (Admin Operations & Comparison)


In [17]:
def admin_required(f):
    def wrap(*args, **kwargs):
        if session.get('role') != 'ADMIN':
            return "Unauthorized", 403
        return f(*args, **kwargs)
    wrap.__name__ = f.__name__
    return wrap

@app.route('/admin/triage')
@login_required
@admin_required
def admin_triage():
    conn = get_db()
    c = conn.cursor()
    c.execute("SELECT * FROM complaints WHERE status != 'RESOLVED' ORDER BY priority_score DESC")
    complaints = c.fetchall()
    conn.close()
    return render_template('admin_triage.html', complaints=complaints)

@app.route('/admin/compare/<int:id1>/<int:id2>')
@login_required
@admin_required
def admin_compare(id1, id2):
    conn = get_db()
    c = conn.cursor()
    c.execute("SELECT * FROM complaints WHERE id = ?", (id1,))
    issue1 = dict(c.fetchone())
    c.execute("SELECT * FROM complaints WHERE id = ?", (id2,))
    issue2 = dict(c.fetchone())
    conn.close()
    
    rationale = compare_issues(issue1, issue2)
    return jsonify({"rationale": rationale})
    
@app.route('/admin/status-update/<int:cid>', methods=['POST'])
@login_required
@admin_required
def admin_status_update(cid):
    new_status = request.form['status']
    conn = get_db()
    c = conn.cursor()
    c.execute("UPDATE complaints SET status = ? WHERE id = ?", (new_status, cid))
    conn.commit()
    conn.close()
    flash("Status updated.")
    return redirect(url_for('admin_triage'))

print("✅ Cell 7 Executed: Admin Triage mapped.")


✅ Cell 7 Executed: Admin Triage mapped.


### CELL 8: UI Template Builder (Injecting HTML into /templates)


In [18]:
import os

os.makedirs('templates', exist_ok=True)
os.makedirs('static/js', exist_ok=True)

base_html = """<!DOCTYPE html>
<html lang="en" class="bg-gray-50 text-gray-900">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>ECPS</title>
    <script src="https://unpkg.com/@tailwindcss/browser@4"></script>
    <style type="text/tailwindcss">
        @theme {
            --color-primary: #1d4ed8;
            --color-critical: #dc2626;
            --color-high: #ea580c;
            --color-medium: #d97706;
            --color-low: #059669;
        }
    </style>
</head>
<body class="min-h-screen flex flex-col">
    <nav class="bg-primary text-white shadow-md p-4 flex justify-between items-center">
        <h1 class="text-xl font-bold tracking-tight">ECPS Triage Engine</h1>
        <div>
            {% if session.get('username') %}
                <span class="mr-4">Welcome, {{ session['username'] }}</span>
                <a href="/logout" class="bg-white text-primary px-3 py-1 rounded-sm text-sm font-medium">Logout</a>
            {% else %}
                <a href="/login" class="mr-4 text-sm font-medium">Login</a>
                <a href="/register" class="text-sm font-medium">Register</a>
            {% endif %}
        </div>
    </nav>
    <main class="flex-grow p-6">
        {% with messages = get_flashed_messages() %}
            {% if messages %}
                <div class="mb-4">
                {% for msg in messages %}
                    <div class="bg-blue-100 text-blue-900 px-4 py-2 rounded-md mb-2 border border-blue-200">
                        {{ msg }}
                    </div>
                {% endfor %}
                </div>
            {% endif %}
        {% endwith %}
        {% block content %}{% endblock %}
    </main>
</body>
</html>"""

with open('templates/base.html', 'w') as f: f.write(base_html)

login_html = """{% extends "base.html" %}
{% block content %}
<div class="max-w-md mx-auto bg-white p-8 rounded-lg shadow-sm border border-gray-100">
    <h2 class="text-2xl font-bold mb-6 text-center">Login</h2>
    <form method="POST">
        <div class="mb-4">
            <label class="block text-sm font-medium mb-1">Username</label>
            <input type="text" name="username" class="w-full border p-2 rounded-md" required>
        </div>
        <div class="mb-4">
            <label class="block text-sm font-medium mb-1">Password</label>
            <input type="password" name="password" class="w-full border p-2 rounded-md" required>
        </div>
        <div class="mb-6">
            <label class="block text-sm font-medium mb-1" id="captcha-question">CAPTCHA: Loading...</label>
            <input type="text" name="captcha" class="w-full border p-2 rounded-md" required>
        </div>
        <button type="submit" class="w-full bg-primary text-white p-2 rounded-md font-semibold shadow-sm hover:bg-blue-800 transition">Log In</button>
    </form>
</div>
<script>
fetch('/captcha').then(r=>r.json()).then(d=>{
    document.getElementById('captcha-question').innerText = "CAPTCHA: " + d.question + " = ?";
});
</script>
{% endblock %}"""

with open('templates/login.html', 'w') as f: f.write(login_html)

register_html = """{% extends "base.html" %}
{% block content %}
<div class="max-w-md mx-auto bg-white p-8 rounded-lg shadow-sm border border-gray-100">
    <h2 class="text-2xl font-bold mb-6 text-center">Student Registration</h2>
    <form method="POST">
        <div class="mb-4">
            <label class="block text-sm font-medium mb-1">Username</label>
            <input type="text" name="username" class="w-full border p-2 rounded-md" required>
        </div>
        <div class="mb-4">
            <label class="block text-sm font-medium mb-1">Email</label>
            <input type="email" name="email" class="w-full border p-2 rounded-md" required>
        </div>
        <div class="mb-6">
            <label class="block text-sm font-medium mb-1">Password</label>
            <input type="password" name="password" class="w-full border p-2 rounded-md" required>
        </div>
        <button type="submit" class="w-full bg-primary text-white p-2 rounded-md font-semibold shadow-sm hover:bg-blue-800 transition">Register</button>
    </form>
</div>
{% endblock %}"""

with open('templates/register.html', 'w') as f: f.write(register_html)

student_html = """{% extends "base.html" %}
{% block content %}
<div class="grid grid-cols-1 md:grid-cols-2 gap-8">
    <div class="bg-white p-6 rounded-lg shadow-sm border border-gray-100">
        <h3 class="text-xl font-bold mb-4">Report an Issue</h3>
        <form method="POST" id="complaintForm">
            <div class="grid grid-cols-2 gap-4 mb-4">
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Title</label>
                    <input type="text" name="title" class="w-full border p-2 rounded-md text-sm" required>
                </div>
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Category</label>
                    <select name="category" id="cat-select" class="w-full border p-2 rounded-md text-sm">
                        <option value="ELECTRICITY">Electricity</option>
                        <option value="WATER">Water / Sanitation</option>
                        <option value="CLASSROOM">Classroom / Structural</option>
                        <option value="TRANSPORT">Transport</option>
                        <option value="INTERNET">Internet / Network</option>
                    </select>
                </div>
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Location</label>
                    <input type="text" name="location" class="w-full border p-2 rounded-md text-sm" required>
                </div>
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Severity (1-5)</label>
                    <input type="number" name="severity" min="1" max="5" class="w-full border p-2 rounded-md text-sm" required>
                </div>
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Affected Headcount</label>
                    <input type="number" name="affected_count" min="0" class="w-full border p-2 rounded-md text-sm" required>
                </div>
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Safety Impact (0-4)</label>
                    <input type="number" name="safety_impact" id="safe-select" min="0" max="4" class="w-full border p-2 rounded-md text-sm" required>
                </div>
                <div>
                    <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Duration (Hours)</label>
                    <input type="number" name="duration_hours" min="1" max="24" class="w-full border p-2 rounded-md text-sm" required>
                </div>
                <div class="flex items-end mb-2">
                    <label class="flex items-center text-sm font-medium">
                        <input type="checkbox" name="is_recurring" class="mr-2 h-4 w-4">
                        Recurring Issue?
                    </label>
                </div>
            </div>
            <div class="mb-4">
                <label class="block text-xs font-semibold uppercase text-gray-500 mb-1">Description</label>
                <textarea name="description" class="w-full border p-2 rounded-md text-sm h-24" required></textarea>
            </div>
            <button type="submit" class="bg-gray-900 text-white px-4 py-2 rounded-md font-semibold text-sm shadow-sm hover:bg-gray-800">Submit Priority Request</button>
        </form>
    </div>
    
    <div>
        <h3 class="text-xl font-bold mb-4">My Dashboard</h3>
        <div class="space-y-4">
            {% for c in complaints %}
                <div class="bg-white p-4 rounded-lg shadow-sm border border-gray-100 flex flex-col gap-2">
                    <div class="flex justify-between">
                        <span class="font-bold">{{ c['title'] }}</span>
                        <span class="text-xs font-bold px-2 py-1 rounded-full bg-gray-100 text-gray-600">{{ c['status'] }}</span>
                    </div>
                    <p class="text-xs text-gray-500">{{ c['assigned_department'] }} | Priority: <span class="font-bold">{{ c['priority_level'] }}</span> ({{ "%.1f"|format(c['priority_score']) }})</p>
                    <div class="mt-2 text-sm p-3 bg-gray-50 rounded-md border border-gray-100 text-gray-700 italic border-l-4 border-l-primary">
                        {{ c['explanation_summary'] }}
                    </div>
                </div>
            {% else %}
                <p class="text-sm text-gray-500">No reported issues.</p>
            {% endfor %}
        </div>
    </div>
</div>
<script src="/static/js/validation.js"></script>
{% endblock %}"""

with open('templates/student_dashboard.html', 'w') as f: f.write(student_html)

admin_html = """{% extends "base.html" %}
{% block content %}
<div class="mb-6 flex justify-between items-center">
    <h2 class="text-2xl font-bold">Admin Triage Dashboard</h2>
</div>
<div class="grid grid-cols-1 lg:grid-cols-3 gap-6">
    <div class="lg:col-span-2 space-y-4">
        {% for c in complaints %}
            <div class="bg-white rounded-lg shadow-sm border-l-4 p-4 flex gap-4 
            {% if c['priority_level'] == 'CRITICAL' %}border-critical{% elif c['priority_level'] == 'HIGH' %}border-high{% elif c['priority_level'] == 'MEDIUM' %}border-medium{% else %}border-low{% endif %}">
                <div class="flex-grow">
                    <div class="flex items-center gap-2 mb-1">
                        <span class="font-bold">#{{ c['id'] }} - {{ c['title'] }}</span>
                        <span class="text-xs bg-gray-100 px-2 py-0.5 rounded-md font-medium text-gray-600">{{ c['category'] }}</span>
                    </div>
                    <p class="text-xs text-gray-500 mb-2">Dept: {{ c['assigned_department'] }} | Location: {{ c['location'] }}</p>
                    <p class="text-sm text-gray-800">{{ c['explanation_summary'] }}</p>
                </div>
                <div class="flex flex-col justify-between items-end min-w-32">
                    <div class="text-right">
                        <div class="text-lg font-black">{{ c['priority_level'] }}</div>
                        <div class="text-xs font-semibold text-gray-500">Score: {{ "%.2f"|format(c['priority_score']) }}</div>
                    </div>
                    <form method="POST" action="/admin/status-update/{{ c['id'] }}" class="mt-2">
                        <select name="status" onchange="this.form.submit()" class="text-xs border p-1 rounded-sm bg-gray-50">
                            <option value="LOGGED" {% if c['status'] == 'LOGGED' %}selected{% endif %}>LOGGED</option>
                            <option value="IN_TRIAGE" {% if c['status'] == 'IN_TRIAGE' %}selected{% endif %}>IN TRIAGE</option>
                            <option value="DISPATCHED" {% if c['status'] == 'DISPATCHED' %}selected{% endif %}>DISPATCHED</option>
                            <option value="RESOLVED" {% if c['status'] == 'RESOLVED' %}selected{% endif %}>RESOLVED</option>
                        </select>
                    </form>
                </div>
            </div>
        {% endfor %}
    </div>
    
    <div class="bg-gray-900 text-gray-100 p-6 rounded-lg shadow-sm h-fit">
        <h3 class="text-lg font-bold mb-4 border-b border-gray-700 pb-2">Pairwise Comparator</h3>
        <p class="text-xs text-gray-400 mb-4">Enter two issue IDs to evaluate prioritization justification.</p>
        <div class="flex gap-2 mb-4">
            <input type="number" id="id1" placeholder="Issue ID A" class="w-1/2 p-2 rounded-md bg-gray-800 border border-gray-700 text-sm">
            <input type="number" id="id2" placeholder="Issue ID B" class="w-1/2 p-2 rounded-md bg-gray-800 border border-gray-700 text-sm">
        </div>
        <button onclick="compareIssues()" class="w-full bg-primary hover:bg-blue-600 text-white px-4 py-2 rounded-md font-semibold text-sm transition">Compare</button>
        <div id="comparisonResult" class="mt-4 p-3 bg-gray-800 rounded-md text-sm border-l-4 border-blue-500 hidden leading-relaxed">
        </div>
    </div>
</div>
<script src="/static/js/comparison.js"></script>
{% endblock %}"""

with open('templates/admin_triage.html', 'w') as f: f.write(admin_html)

val_js = """document.getElementById('cat-select')?.addEventListener('change', (e) => {
    let s = document.getElementById('safe-select');
    if (e.target.value === 'INTERNET') { s.max = 1; s.value = 0; }
    else if (e.target.value === 'TRANSPORT') { s.max = 2; if(s.value>2) s.value=2; }
    else if (e.target.value === 'WATER') { s.max = 3; if(s.value>3) s.value=3; }
    else { s.max = 4; }
});"""
with open('static/js/validation.js', 'w') as f: f.write(val_js)

comp_js = """function compareIssues() {
    const id1 = document.getElementById('id1').value;
    const id2 = document.getElementById('id2').value;
    if(!id1 || !id2) return;
    fetch(`/admin/compare/${id1}/${id2}`)
        .then(r => r.json())
        .then(d => {
            let box = document.getElementById('comparisonResult');
            box.innerText = d.rationale;
            box.classList.remove('hidden');
        }).catch(err => {
            let box = document.getElementById('comparisonResult');
            box.innerText = "Error fetching comparison or IDs do not exist.";
            box.classList.remove('hidden');
        });
}"""
with open('static/js/comparison.js', 'w') as f: f.write(comp_js)

print("✅ Cell 8 Executed: UI Templates and JS generated.")


✅ Cell 8 Executed: UI Templates and JS generated.


### CELL 9: Verification Unit Tests (Algorithm & Validation)


In [19]:
print("Running Internal Verification Tests...")

# TC-02: Contradiction Trap
is_valid, msg = validate_complaint("INTERNET", 5, 100, 4, "Library", "Internet down.")
assert not is_valid, "Failed TC-02"
print("✓ TC-02 Passed: Internet outage with high safety impact correctly rejected.")

# TC-01: Severe Hazard vs Mass Inconvenience
score_a = compute_priority("ELECTRICITY", 5, 4, 4, 2, 0)
score_b = compute_priority("INTERNET", 5, 300, 0, 2, 0)
assert score_a > score_b, "Failed TC-01"
print(f"✓ TC-01 Passed: Score A ({score_a:.2f}) > Score B ({score_b:.2f})")

print("✅ Cell 9 Executed: All Unit Tests Passed.")


Running Internal Verification Tests...
✓ TC-02 Passed: Internet outage with high safety impact correctly rejected.
✓ TC-01 Passed: Score A (20.58) > Score B (7.21)
✅ Cell 9 Executed: All Unit Tests Passed.


### CELL 10: Server Runner & Gateway


In [ ]:
print("Starting ECPS Server...")
print("Open http://127.0.0.1:5000 in your browser.")
print("Login with Admin Credentials -> user: admin | pass: admin123")

# Run flask in a background thread to allow notebook to finish cell
def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)
    
server_thread = threading.Thread(target=run_app)
server_thread.daemon = True
server_thread.start()

print("✅ Cell 10 Executed: Server Thread running in background.")


Starting ECPS Server...
Open http://127.0.0.1:5000 in your browser.
Login with Admin Credentials -> user: admin | pass: admin123
✅ Cell 10 Executed: Server Thread running in background.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.26.5.111:5000
Press CTRL+C to quit
